In [ ]:
# Core
import os
import pandas as pd
import copy
import numpy as np
import pybedtools

# Graphs
import matplotlib.pyplot as plt
import seaborn as sns
# Statistics
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import binomtest
from adjustText import adjust_text
from sklearn.metrics import precision_recall_curve
from scipy.stats import spearmanr
from scipy.stats import pearsonr

from scipy.stats import ks_2samp # Kolmogorov–Smirnov (KS)
from scipy.stats import linregress
from scipy.stats import gaussian_kde


In [ ]:
import os, sys
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))
import const
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
from const import save_fig
const.set_plot_style()

# --- Base directory (EDIT THIS) ---
# Paths below are relative to this lab-data root (the Collaboration directory).
BASE_DIR = '/home/labs/davidgo/Collaboration'   # <- set to your local copy
os.chdir(BASE_DIR)

In [ ]:
# Parameters
filter_for_oligos_w_1_variant = True

# Plotting functions

In [ ]:
fig_dir ='humanMPRA/TF_analysis/final/figures'
dataset_dir = 'humanMPRA/TF_analysis/final/datasets_for_paper'

In [ ]:
figure_size = (8,6)
add_labels = True
size_legend = True

def plot_density_scatter(xvals, yvals, title):
    mask = ~np.isnan(xvals) & ~np.isnan(yvals)
    xy = np.vstack([xvals[mask], yvals[mask]])
    z = gaussian_kde(xy)(xy)

    idx = z.argsort()
    x_sorted, y_sorted, z_sorted = xvals[mask][idx], yvals[mask][idx], z[idx]

    plt.scatter(
        x_sorted, y_sorted,
        c=z_sorted, cmap=custom_cmap, s=10, alpha=0.5, edgecolor='none'
    )

    plt.xlabel(r'PBM z-score')
    plt.ylabel(r'FIMO z-score')
    slope, intercept, r_value, p_value, std_err = linregress(xvals[mask], yvals[mask])
    x_fit = np.linspace(xvals[mask].min(), xvals[mask].max(), 100)
    y_fit = slope * x_fit + intercept
    plt.plot(x_fit, y_fit, color='black', lw=2, label='Regression line')
    plt.title(title)
    plt.show()

def plot_TF_jaccard_similarity_heatmap(TF_names, merged_TF_MPRA_df):
    enriched_gene_positions = {}
    for tf in TF_names:
        positions = set(merged_TF_MPRA_df.loc[merged_TF_MPRA_df['motif_id_clean'] == tf, 'seq_id'])
        enriched_gene_positions[tf] = positions

    similarity_matrix = pd.DataFrame(index=TF_names, columns=TF_names, dtype=float)
    from itertools import combinations
    for tf1, tf2 in combinations(TF_names, 2):
        set1, set2 = enriched_gene_positions[tf1], enriched_gene_positions[tf2]
        intersection = len(set1 & set2)
        union = len(set1 | set2)
        jaccard = intersection / union if union != 0 else 0.0
        similarity_matrix.loc[tf1, tf2] = jaccard
        similarity_matrix.loc[tf2, tf1] = jaccard

    np.fill_diagonal(similarity_matrix.values, 1.0)
    similarity_matrix = similarity_matrix.fillna(0)
    annot_values = similarity_matrix.round(2)

    g = sns.clustermap(similarity_matrix.astype(float),
                       cmap="Blues",
                       fmt=".2f",
                       annot_kws={"size": 12},
                       figsize=(16, 12),
                       cbar_kws={'label': 'Jaccard Similarity'},
                       linewidths=0.5)

    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=16, rotation=90)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=16, rotation=0)
    g.fig.suptitle("Clustered TF Similarity by Genomic Position, all enriched TFs", y=1.02, fontsize=20)
    g.fig.subplots_adjust(top=0.93)
    return g.fig

def plot_TF_enrichment_volcano(xvals, yvals, size, labels, title,scaling_factor=1):
    plt.figure(figsize=figure_size)
    plt.scatter(xvals, yvals, color='grey', alpha=0.7, s=size/scaling_factor)
    if add_labels:
        significant = yvals > -np.log10(0.05)
        for i, txt in enumerate(labels[significant]):
            plt.text(xvals[significant].iloc[i], yvals[significant].iloc[i], txt, fontsize=8)
    if size_legend:
        for s in [10, 50, 100]:
            plt.scatter([], [], color='grey', alpha=0.7, s=s/scaling_factor, label=f'{s} seqs')
        plt.legend(title='TF binding sites', loc='upper left')

    plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='FDR=0.05')
    plt.axvline(0, color='black', linestyle=':')
    plt.xlabel('log2(Odds Ratio)')
    plt.ylabel('-log10(FDR)')
    plt.title(title)
    plt.legend()
    plt.tight_layout()

def plot_Pearson_volcano(xvals, yvals, size, labels, title,scaling_factor=1):
    plt.figure(figsize=figure_size)
    plt.scatter(xvals, yvals, color='grey', alpha=0.7, s=size/scaling_factor)
    if add_labels:
        significant = yvals > -np.log10(0.05)
        for i, txt in enumerate(labels[significant]):
            plt.text(xvals[significant].iloc[i], yvals[significant].iloc[i], txt, fontsize=8)
    if size_legend:
        for s in [10, 50, 100]:
            plt.scatter([], [], color='grey', alpha=0.7, s=s/scaling_factor, label=f'{s} seqs')
        plt.legend(title='TF binding sites', loc='upper left')
           
    plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='FDR=0.05')
    plt.axvline(0, color='black', linestyle=':')
    plt.xlabel("Pearson's correlation coefficient")
    plt.ylabel('-log10(FDR)')
    plt.title(title)
    plt.legend()
    plt.tight_layout()

def plot_TF_binding_vs_LFC(tf_mpra_merged,diff_binding_col,LFC_col,diff_active_osteoblasts_col, tf,just_diff_active=False):
    """
    Plot TF binding vs oligo LFC with 2D density and scatter overlay.

    Args:
        tf_mpra_merged (pd.DataFrame): Merged TF and MPRA data
        tf (str): TF name to plot
    """
    
    relevant_data = tf_mpra_merged[tf_mpra_merged['motif_id_clean'] == tf].copy()
    if just_diff_active:
        relevant_data = relevant_data[relevant_data[diff_active_osteoblasts_col] == 1]
    else:
        relevant_data = relevant_data[~relevant_data[LFC_col].isna()]

    motif = tf

    fig, ax = plt.subplots(figsize=(7, 6))

    # Calculate Pearson correlation
    if len(relevant_data) > 1:
        correlation, p_value = pearsonr(
            relevant_data[diff_binding_col], relevant_data[LFC_col]
        )
    else:
        correlation = np.nan
        p_value = np.nan

    # 2D KDE (density)
    sns.kdeplot(
        data=relevant_data,
        x=diff_binding_col,
        y=LFC_col,
        fill=True,
        thresh=0.05,
        cmap='Reds',
        levels=100,
        ax=ax
    )

    # Overlay scatter plot with color mapping
    sns.scatterplot(
        data=relevant_data,
        x=diff_binding_col,
        y=LFC_col,
        hue=diff_active_osteoblasts_col,
        palette={True: 'green', False: 'grey'},
        alpha=0.7,
        s=20,
        ax=ax
    )

    # Reference lines
    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)

    # Axis labeling
    ax.set_xlabel(f'{motif} diff binding')
    ax.set_ylabel('LFC of oligos (derived vs ancestral)')

    # Update the title with Pearson correlation
    if not np.isnan(correlation):
        ax.set_title(
            f'{motif} binding vs oligo LFC\n(Scatter over 2D Density, all data)\n'
            f'(n={len(relevant_data)}) | Pearson r={correlation:.2f}, p={p_value:.2e}'
        )
    else:
        ax.set_title(
            f'{motif} binding vs oligo LFC\n(Scatter over 2D Density, all data)\n'
            f'(n={len(relevant_data)})'
        )

    # Remove gridlines
    ax.grid(False)
    # Adjust legend
    handles, labels = ax.get_legend_handles_labels()
    new_labels = ['Not differential', 'Differential'] if labels[0] == 'False' else ['Differential', 'Not differential']
    ax.legend(handles, new_labels, title='Activity type')

    ax.set_xlim(-5, 5)
    ax.set_ylim(-2, 2)

    plt.tight_layout()

## Read relevant files

In [ ]:
# Load MPRA results
MPRA_output= pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     header=0)

MPRA_output.rename(columns={'oligo':'seq_id',}, inplace=True)

# Add fields
MPRA_output['active_bool'] = MPRA_output['differential_activity'].isin([True, False])
MPRA_output['diff_active_bool'] = MPRA_output['differential_activity']==True

#keep only relevant fields
MPRA_output = MPRA_output[['seq_id','variants_genomic','variants_count','logFC_derived_vs_ancestral','activity_fdr_ancestral',
                           'activity_fdr_derived','differential_activity_fdr','active_bool','diff_active_bool']]


if filter_for_oligos_w_1_variant:
    print(f"Filtering for oligos with exactly 1 variant. Original count: {len(MPRA_output)}, Filtered count: {len(MPRA_output[MPRA_output['variants_count'] == 1])}")
    MPRA_output = MPRA_output[MPRA_output['variants_count'] == 1]


In [ ]:
# Load FIMO and PBM results
hMPRA_PBM = pd.read_csv(f'humanMPRA/TF_analysis/final/results/hMPRA_PBM_oligo_TF_merged.tsv', sep='\t')
hMPRA_PBM['motif_id_clean'] = hMPRA_PBM['motif_id_clean'].str.upper()

hMPRA_FIMO = pd.read_csv(f'humanMPRA/TF_analysis/final/results/hMPRA_FIMO_oligo_TF_merged.tsv', sep='\t')
hMPRA_FIMO['motif_id_clean'] = hMPRA_FIMO['motif_id_clean'].str.upper()


In [ ]:
all_motifs = pd.concat([hMPRA_PBM, hMPRA_FIMO], axis = 0)

In [ ]:
merged_df = pd.merge(all_motifs, MPRA_output, on='seq_id', how='outer')
PBM_merged_df = pd.merge(hMPRA_PBM, MPRA_output, on='seq_id', how='outer')
FIMO_merged_df = pd.merge(hMPRA_FIMO, MPRA_output, on='seq_id', how='outer')

# Validation - correlation between FIMO and PBM

In [ ]:
# get motif lists
pbm_motifs = hMPRA_PBM['motif_id_clean'].dropna().unique().tolist()
fimo_motifs = hMPRA_FIMO['motif_id_clean'].dropna().unique().tolist()
shared_motifs = sorted(list(set(pbm_motifs) & set(fimo_motifs)))

print(f"Number of PBM motifs: {len(pbm_motifs)}")
print(f"Number of FIMO motifs: {len(fimo_motifs)}")
print(f"Number of shared motifs: {len(shared_motifs)}")

In [ ]:
# merge PBM and FIMO z-scores by seq_id + motif_id_clean
pbm_z = PBM_merged_df[['seq_id','motif_id_clean','diff_binding_zscore']].rename(columns={'diff_binding_zscore':'z_pbm'})
fimo_z = FIMO_merged_df[['seq_id','motif_id_clean','diff_binding_zscore']].rename(columns={'diff_binding_zscore':'z_fimo'})
merged_pf = pd.merge(pbm_z, fimo_z, on=['seq_id','motif_id_clean'], how='inner')

# keep only shared motifs and drop rows with missing z-scores
merged_pf = merged_pf[merged_pf['motif_id_clean'].isin(shared_motifs)].dropna(subset=['z_pbm','z_fimo'])

# compute spearman correlation per motif
rows = []
for motif, grp in merged_pf.groupby('motif_id_clean'):
    a = grp['z_pbm'].values
    b = grp['z_fimo'].values
    n = len(a)
    if n == 0:
        rho, p = (np.nan, np.nan)
    else:
        rho, p = spearmanr(a, b)
    rows.append((motif, n, rho, p))

corr_df = pd.DataFrame(rows, columns=['motif_id_clean','n_pairs','spearman_rho','pval']).sort_values('pval')


In [ ]:
plot_density_scatter(merged_pf['z_pbm'].values, merged_pf['z_fimo'].values, 'PBM vs FIMO z-score correlation')
save_fig(plt, 'PBM_FIMO_zscore_correlation_osteobalsts', fig_dir)
plt.show()

# PBM

## Jaccared Similarity Index

In [ ]:
plot = plot_TF_jaccard_similarity_heatmap(pbm_motifs, PBM_merged_df)
save_fig(plot, 'PBM_jaccard_similarity_osteobalsts', fig_dir)
plt.show()

## motif enrichment in diff-active oligos

In [ ]:
def run_tf_fisher_test(
    relevant_motifs,
    relevant_df,
    diff_active_col,
    min_a_bind_diff=0
):
    results = []

    for tf in relevant_motifs:
        bind_df = relevant_df[relevant_df["motif_id_clean"] == tf]
        bind_tf_and_diff_active = bind_df.loc[
            bind_df[diff_active_col] == 1, "seq_id"
        ].nunique()
        bind_tf_and_active = bind_df.loc[
            bind_df[diff_active_col] == 0, "seq_id"
        ].nunique()

        # skip TFs with too few bind+diff_active seqs
        if bind_tf_and_diff_active < min_a_bind_diff:
            continue

        not_bind_df = relevant_df[relevant_df["motif_id_clean"] != tf]
        not_bind_tf_and_diff_active = not_bind_df.loc[
            not_bind_df[diff_active_col] == 1, "seq_id"
        ].nunique()
        not_bind_tf_and_active = not_bind_df.loc[
            not_bind_df[diff_active_col] == 0, "seq_id"
        ].nunique()

        a = bind_tf_and_diff_active
        b = bind_tf_and_active
        c = not_bind_tf_and_diff_active
        d = not_bind_tf_and_active

        table = [[a, b], [c, d]]
        oddsratio, pvalue = fisher_exact(table)

        results.append({
            "TF": tf,
            "a_bind_diff": a,
            "b_bind_notdiff": b,
            "c_notbind_diff": c,
            "d_notbind_notdiff": d,
            "odds_ratio": oddsratio,
            "pvalue": pvalue
        })

    fisher_df = pd.DataFrame(results)

    if not fisher_df.empty:
        fisher_df["pval_fdr"] = multipletests(
            fisher_df["pvalue"].values,
            method="fdr_bh"
        )[1]
        fisher_df = fisher_df.sort_values("pval_fdr").reset_index(drop=True)
    else:
        fisher_df["pval_fdr"] = pd.Series(dtype=float)

    return fisher_df


def run_tf_diffactive_correlation(
    relevant_dataset,
    relevant_motifs,
    diff_active_col,
    logFC_col,
    min_valid_oligos=2
):
    corr_results = []

    for tf in relevant_motifs:
        tf_df = relevant_dataset[relevant_dataset["motif_id_clean"] == tf]
        tf_df = tf_df[tf_df[diff_active_col] == 1]

        # keep only rows with non-null diff_binding_zscore and chosen logFC column
        valid = tf_df[["diff_binding_zscore", logFC_col]].dropna()
        n = len(valid)

        if n >= min_valid_oligos and n > 1:
            rho, pval = pearsonr(valid["diff_binding_zscore"], valid[logFC_col])
        else:
            rho, pval = (np.nan, np.nan)

        corr_results.append({
            "TF": tf,
            "n_pairs": n,
            "pearson_rho": rho,
            "pvalue_pearson": pval
        })

    corr_df = pd.DataFrame(corr_results)

    valid_pvals = corr_df["pvalue_pearson"].notna()
    corr_df["pval_fdr_pearson"] = np.nan

    if valid_pvals.any():
        corr_df.loc[valid_pvals, "pval_fdr_pearson"] = multipletests(
            corr_df.loc[valid_pvals, "pvalue_pearson"].values,
            method="fdr_bh"
        )[1]

    corr_df = corr_df.sort_values("pval_fdr_pearson", na_position="last").reset_index(drop=True)

    return corr_df

In [ ]:
fisher_params = {
    "relevant_motifs": pbm_motifs,
    "relevant_df": PBM_merged_df,
    "diff_active_col": "diff_active_bool",
    "min_a_bind_diff": 0
}

fisher_PBM_df = run_tf_fisher_test(**fisher_params)
fisher_PBM_df.to_csv(f'{dataset_dir}/PBM_osteo_TF_enrichment_diff_active.tsv', sep='\t', index=False)


In [ ]:
plot_TF_enrichment_volcano(np.log2(fisher_PBM_df['odds_ratio']),
 -np.log10(fisher_PBM_df['pval_fdr']),
  fisher_PBM_df['a_bind_diff'],
   fisher_PBM_df['TF'],
    'Volcano plot of TF enrichment (PBM)',
    scaling_factor=3)

save_fig(plt, 'PBM_TF_enrichment_diff_active_volcano', fig_dir)

plt.show()


## Correlation of diff activity with diff binding

In [ ]:
corr_params = {
    "relevant_dataset": PBM_merged_df,
    "relevant_motifs": pbm_motifs,
    "diff_active_col": "diff_active_bool",
    "logFC_col": "logFC_derived_vs_ancestral",
    "min_valid_oligos": 0
}

pearson_PBM_df = run_tf_diffactive_correlation(**corr_params)
pearson_PBM_df.to_csv('ArchaicDerivedMPRA/TF_analysis/for_ryder/pearson_correlation_diff_active_diff_binding/pearson_PBM_osteo_TF_diff_active_diff_binding.tsv', sep='\t', index=False)
pearson_PBM_df

In [ ]:
plot_Pearson_volcano(pearson_PBM_df['pearson_rho'],
                       -np.log10(pearson_PBM_df['pval_fdr_pearson']),
                         pearson_PBM_df['n_pairs'],
                           pearson_PBM_df['TF'],
                             'TF vs logFC_derived_vs_ancestral correlation volcano plot')

save_fig(plt, 'PBM_TF_pearson_correlation_volcano', fig_dir)
plt.show()

## Main figure

In [ ]:
def plot_TF_enrichment_volcano_colored(xvals, yvals, size, labels, significantly_correlated, title,scaling_factor=1,legend_sizes=[10, 50, 100]):
    x = np.asarray(xvals)
    y = np.asarray(yvals)
    labels_arr = np.asarray(labels)
    sig_spearman = np.asarray(significantly_correlated).astype(bool)

    colors = np.where(sig_spearman, 'green', 'grey')

    plt.figure(figsize=figure_size)
    plt.scatter(x, y, c=colors, linewidths=0, alpha=0.7, s=np.asarray(size)/scaling_factor)

    if add_labels:
        sig_fdr = y > -np.log10(0.05)
        texts = []
        for xi, yi, txt, c in zip(x[sig_fdr], y[sig_fdr], labels_arr[sig_fdr], colors[sig_fdr]):
            texts.append(plt.text(xi, yi, txt, fontsize=8, color=c))

        if texts:
            adjust_text(
                texts,
                only_move={'points': 'y', 'text': 'xy'},
                arrowprops=dict(arrowstyle='-', color= c,alpha = 0.7)  # <- arrow color
            )

    if size_legend:
        for s in legend_sizes:
            plt.scatter([], [], color='grey', alpha=0.7, s=s/scaling_factor, label=f'{s} seqs')
        plt.legend(title='TF binding sites', loc='upper left')

    # color legend
    #plt.scatter([], [], c='turquoise', label='Spearman TRUE')
    #plt.scatter([], [], c='grey', label='Spearman FALSE')
    plt.legend()

    #plt.xticks([-2,0,2])
    #plt.yticks([0,7])

    plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='FDR=0.05')
    plt.axvline(0, color='black', linestyle=':')
    plt.xlabel('log2(Odds Ratio)')
    plt.ylabel('-log10(FDR)')
    plt.title(title)
    plt.tight_layout()

    #save_fig(plt, 'osteoblasts_TFs_enriched_in_diff_active_oligos', 'ArchaicDerivedMPRA/TF_analysis/figures/archaic_osteoblasts')

In [ ]:
fisher_PBM_with_correlation = fisher_PBM_df.merge(pearson_PBM_df[['TF', 'pval_fdr_pearson']], left_on='TF', right_on='TF', how='left')
fisher_PBM_with_correlation['pearson_significant'] = fisher_PBM_with_correlation['pval_fdr_pearson'] < 0.05

plot_TF_enrichment_volcano_colored(np.log2(fisher_PBM_with_correlation['odds_ratio']),
 -np.log10(fisher_PBM_with_correlation['pval_fdr']),
  fisher_PBM_with_correlation['a_bind_diff'],
   fisher_PBM_with_correlation['TF'],
   fisher_PBM_with_correlation['pearson_significant'],
    '',
    scaling_factor=8,
    legend_sizes=[100, 200, 500])

pearson_PBM_df.to_csv(dataset_dir + '/pearson_PBM_TF_diff_active_diff_binding.tsv', sep='\t', index=False)
save_fig(plt, 'main_figure_PBM', fig_dir)
plt.show()


# FIMO

## Jaccard Similarity Index

In [ ]:
plot = plot_TF_jaccard_similarity_heatmap(fimo_motifs, FIMO_merged_df)
#save_fig(plot, 'FIMO_jaccard_similarity_osteobalsts', 'ArchaicDerivedMPRA/TF_analysis/figures/archaic_osteoblasts')
plt.show()

## motif enrichment in diff-active oligos

In [ ]:
fisher_params = {
    "relevant_motifs": fimo_motifs,
    "relevant_df": FIMO_merged_df,
    "diff_active_col": "diff_active_bool",
    "min_a_bind_diff": 0
}

fisher_FIMO_df = run_tf_fisher_test(**fisher_params)
fisher_FIMO_df.to_csv(dataset_dir + '/fimo_TF_enrichment_diff_active.tsv', sep='\t', index=False)

In [ ]:
plot_TF_enrichment_volcano(np.log2(fisher_FIMO_df['odds_ratio']),
 -np.log10(fisher_FIMO_df['pval_fdr']),
  fisher_FIMO_df['a_bind_diff']*2,
   fisher_FIMO_df['TF'],
    'Volcano plot of TF enrichment (FIMO)',
    scaling_factor=8)

save_fig(plt, 'FIMO_TF_enrichment_diff_active_volcano', fig_dir)
plt.show()


## Correlation of diff activity with diff binding

In [ ]:
corr_params = {
    "relevant_dataset": FIMO_merged_df,
    "relevant_motifs": fimo_motifs,
    "diff_active_col": "diff_active_bool",
    "logFC_col": "logFC_derived_vs_ancestral",
    "min_valid_oligos": 0
}

pearson_FIMO_df = run_tf_diffactive_correlation(**corr_params)
#pearson_osteo_FIMO_df.to_csv('ArchaicDerivedMPRA/TF_analysis/for_ryder/pearson_correlation_diff_active_diff_binding/pearson_FIMO_osteo_TF_diff_active_diff_binding.tsv', sep='\t', index=False)
pearson_FIMO_df

In [ ]:
pearson_FIMO_df

In [ ]:
plot_Pearson_volcano(pearson_FIMO_df['pearson_rho'],
                       -np.log10(pearson_FIMO_df['pval_fdr_pearson']),
                         pearson_FIMO_df['n_pairs'],
                           pearson_FIMO_df['TF'],
                             'TF vs logFC_derived_vs_ancestral correlation volcano plot',
                             scaling_factor=8)

save_fig(plt, 'FIMO_TF_pearson_correlation_volcano', fig_dir)
plt.show()

In [ ]:
fisher_FIMO_with_correlation = fisher_FIMO_df.merge(pearson_FIMO_df[['TF', 'pval_fdr_pearson']], left_on='TF', right_on='TF', how='left')
fisher_FIMO_with_correlation['pearson_significant'] = fisher_FIMO_with_correlation['pval_fdr_pearson'] < 0.05

plot_TF_enrichment_volcano_colored(np.log2(fisher_FIMO_with_correlation['odds_ratio']),
 -np.log10(fisher_FIMO_with_correlation['pval_fdr']),
  fisher_FIMO_with_correlation['a_bind_diff'],
   fisher_FIMO_with_correlation['TF'],
   fisher_FIMO_with_correlation['pearson_significant'],
    '',
    scaling_factor=1,
    legend_sizes=[10, 50, 100]
    )

pearson_FIMO_df.to_csv(dataset_dir + '/pearson_FIMO_TF_diff_active_diff_binding.tsv', sep='\t', index=False)
save_fig(plt, 'main_figure_FIMO', fig_dir)
plt.show()

In [ ]:
fig_dir

In [ ]:
fisher_FIMO_with_correlation = fisher_FIMO_df.merge(pearson_FIMO_df[['TF', 'pval_fdr_pearson']], left_on='TF', right_on='TF', how='left')

fisher_FIMO_with_correlation_capped = fisher_FIMO_with_correlation.copy()
#cap the y axis at 30
capp = 15
fisher_FIMO_with_correlation_capped['pval_fdr_capped'] = fisher_FIMO_with_correlation_capped['pval_fdr'].apply(lambda x: x if x > 10**-capp else 10**-capp)
fisher_FIMO_with_correlation_capped['pearson_significant'] = fisher_FIMO_with_correlation_capped['pval_fdr_pearson'] < 0.05

plot_TF_enrichment_volcano_colored(np.log2(fisher_FIMO_with_correlation_capped['odds_ratio']),
 -np.log10(fisher_FIMO_with_correlation_capped['pval_fdr_capped']),
  fisher_FIMO_with_correlation_capped['a_bind_diff'],
   fisher_FIMO_with_correlation_capped['TF'],
   fisher_FIMO_with_correlation_capped['pearson_significant'],
    '',
    scaling_factor=1,
    legend_sizes=[10, 50, 100]
    )

save_fig(plt, f'main_figure_FIMO_capped_at_{capp}', fig_dir)
plt.show()

In [ ]:
# Combine FIMO and PBM results and identify TFs significant in all tests
combined_results = fisher_PBM_with_correlation.merge(
    fisher_FIMO_with_correlation,
    on='TF',
    suffixes=('_PBM', '_FIMO')
)

# Filter for TFs significant in all four tests:
# 1. Fisher FDR < 0.05 for PBM
# 2. Fisher FDR < 0.05 for FIMO
# 3. Pearson significant for PBM
# 4. Pearson significant for FIMO
significant_all_tests = combined_results[
    (combined_results['pval_fdr_PBM'] < 0.05) &
    (combined_results['pval_fdr_FIMO'] < 0.05) &
    (combined_results['pearson_significant_PBM'] == True) &
    (combined_results['pearson_significant_FIMO'] == True)
]

print(f"\nTFs significant in all tests (Fisher x2 + Pearson correlation x2):")
print(significant_all_tests[['TF', 'pval_fdr_PBM', 'pval_fdr_FIMO', 'pearson_significant_PBM', 'pearson_significant_FIMO']])
print(f"\nTotal TFs meeting all criteria: {len(significant_all_tests)}")
if len(significant_all_tests) > 0:
    print(f"TF names: {significant_all_tests['TF'].tolist()}")

In [ ]:
pbm_motifs

In [ ]:
plot = plot_TF_jaccard_similarity_heatmap(significant_all_tests['TF'].tolist(), PBM_merged_df)
#save_fig(plot, 'PBM_jaccard_similarity_osteobalsts', fig_dir)
plt.show()

In [ ]:
plot = plot_TF_jaccard_similarity_heatmap(significant_all_tests['TF'].tolist(), FIMO_merged_df)
#save_fig(plot, 'PBM_jaccard_similarity_osteobalsts', fig_dir)
plt.show()

# Specific TFs

In [ ]:
def plot_TF_binding_vs_LFC_for_figure(tf_mpra_merged,diff_binding_col,LFC_col,diff_active_osteoblasts_col,cell_type, tf,just_diff_active=False):
    """
    Plot TF binding vs oligo LFC with 2D density and scatter overlay.

    Args:
        tf_mpra_merged (pd.DataFrame): Merged TF and MPRA data
        tf (str): TF name to plot
    """
    
    relevant_data = tf_mpra_merged[tf_mpra_merged['motif_id_clean'] == tf].copy()
    if just_diff_active:
        relevant_data = relevant_data[relevant_data[diff_active_osteoblasts_col] == 1]
    else:
        relevant_data = relevant_data[~relevant_data[LFC_col].isna()]

    motif = tf

    fig, ax = plt.subplots(figsize=(7, 6))

    # Calculate Pearson correlation
    if len(relevant_data) > 1:
        slope, intercept, r_val, p_val_lin, std_err = linregress(relevant_data[diff_binding_col], relevant_data[LFC_col])
        x_fit = np.linspace(relevant_data[diff_binding_col].min(), relevant_data[diff_binding_col].max(), 100)
        y_fit = slope * x_fit + intercept
        ax.plot(x_fit, y_fit, color='black', linewidth=2, label='_nolegend_')
        correlation, p_value = pearsonr(
            relevant_data[diff_binding_col], relevant_data[LFC_col]
        )
    else:
        correlation = np.nan
        p_value = np.nan

    # Overlay scatter plot with color mapping
    sns.scatterplot(
        data=relevant_data,
        x=diff_binding_col,
        y=LFC_col,
        hue=diff_active_osteoblasts_col,
        palette={True: 'green', False: 'grey'},
        alpha=0.7,
        s=20,
        ax=ax
    )

    # Reference lines
    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)

    # Axis labeling
    ax.set_xlabel(f'{motif} diff binding')
    ax.set_ylabel('LFC of oligos (derived vs ancestral)')

    # Update the title with Pearson correlation
    if not np.isnan(correlation):
        ax.set_title(
            f'{motif} binding vs oligo LFC\n(Scatter, differentially active oligos)\n'
            f'(n={len(relevant_data)}) | Pearson r={correlation:.2f}, p={p_value:.2e}'
        )
    else:
        ax.set_title(
            f'{motif} binding vs oligo LFC\n(Scatter over 2D Density, all data)\n'
            f'(n={len(relevant_data)})'
        )


    # Remove gridlines
    ax.grid(False)
    # Adjust legend
    handles, labels = ax.get_legend_handles_labels()
    new_labels = ['Not differential', 'Differential'] if labels[0] == 'False' else ['Differential', 'Not differential']
    ax.legend(handles, new_labels, title='Activity type')

    ax.set_xlim(-5, 5)
    ax.set_ylim(-1, 1)

    ax.set_xticks([-4,0,4])
    ax.set_yticks([-1,0,1])

    plt.tight_layout()
    #save_fig(plt, f'{cell_type}_{method}_TF_{TF}_binding_vs_LFC', 'ArchaicDerivedMPRA/TF_analysis/figures/specific_motifs')

    return relevant_data
  



# HES1

## PBM Osteo

In [ ]:
TF = 'FOSL1'
method = 'PBM'
cell_type = 'osteo'
if method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_bool'
elif method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_bool'


specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
#specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()

## FIMO Osteo

In [ ]:
TF = 'HES1'
method = 'FIMO'
cell_type = 'osteo'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'


specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
#specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()

## PBM NPCs

In [ ]:
TF = 'HES1'
method = 'PBM'
cell_type = 'npc'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'

specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()

## FIMO NPCs

In [ ]:
TF = 'HES1'
method = 'FIMO'
cell_type = 'npc'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'


specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()

# FOSL1

## PBM Osteo

In [ ]:
TF = 'FOSL1'
method = 'PBM'
cell_type = 'osteo'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'

specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 

plt.show()

## FIMO Osteo

In [ ]:
TF = 'FOSL1'
method = 'FIMO'
cell_type = 'osteo'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'


specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()


## NPCs PBM

In [ ]:
TF = 'FOSL1'
method = 'PBM'
cell_type = 'npc'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'


specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()

## NPCs FIMO

In [ ]:
TF = 'FOSL1'
method = 'FIMO'
cell_type = 'npc'
if cell_type == 'osteo' and method == 'PBM':
    relevant_df = PBM_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'osteo' and method == 'FIMO':
    relevant_df = FIMO_merged_df
    logFC_col = 'logFC_derived_vs_ancestral'
    diff_active_col = 'diff_active_osteoblasts'
elif cell_type == 'npc' and method == 'PBM':
    relevant_df = PBM_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'
elif cell_type == 'npc' and method == 'FIMO':
    relevant_df = FIMO_npc_merged_df
    logFC_col = 'logFC_npc'
    diff_active_col = 'diff_active_npc'


specific_TF_df = plot_TF_binding_vs_LFC_for_figure(relevant_df,'diff_binding_zscore',logFC_col,diff_active_col,cell_type,TF,just_diff_active=True)
specific_TF_df.to_csv(f'ArchaicDerivedMPRA/TF_analysis/for_ryder/specific_motifs/{cell_type}_{method}_{TF}_binding_vs_LFC_scatter_data.tsv', sep='\t', index=False) 
plt.show()

# NPCs leftover

In [ ]:
# merge PBM and FIMO z-scores by seq_id + motif_id_clean
pbm_z = archaic_npc_PBM[['seq_id','motif_id_clean','diff_binding_zscore']].rename(columns={'diff_binding_zscore':'z_pbm'})
fimo_z = archaic_npc_FIMO[['seq_id','motif_id_clean','diff_binding_zscore']].rename(columns={'diff_binding_zscore':'z_fimo'})
merged_pf = pd.merge(pbm_z, fimo_z, on=['seq_id','motif_id_clean'], how='inner')

# keep only shared motifs and drop rows with missing z-scores
merged_pf = merged_pf[merged_pf['motif_id_clean'].isin(shared_npc_motifs)].dropna(subset=['z_pbm','z_fimo'])
# compute spearman correlation per motif
rows = []
for motif, grp in merged_pf.groupby('motif_id_clean'):
    a = grp['z_pbm'].values
    b = grp['z_fimo'].values
    n = len(a)
    if n == 0:
        rho, p = (np.nan, np.nan)
    else:
        rho, p = spearmanr(a, b)
    rows.append((motif, n, rho, p))

corr_df = pd.DataFrame(rows, columns=['motif_id_clean','n_pairs','spearman_rho','pval']).sort_values('pval')

plot_density_scatter(merged_pf['z_pbm'].values, merged_pf['z_fimo'].values, 'PBM vs FIMO z-score correlation')
#save_fig(plt, 'PBM_FIMO_zscore_correlation_npc', 'ArchaicDerivedMPRA/TF_analysis/figures/validation')
plt.show()

In [ ]:
plot_TF_jaccard_similarity_heatmap(pbm_npc_motifs, PBM_npc_merged_df)
#save_fig(plt, 'PBM_jaccard_similarity_npc', 'ArchaicDerivedMPRA/TF_analysis/figures/archaic_npcs')
plt.show()